# 11 Feature-Layer Ablation

bottleneck / skip1 / skip2 / decoder1 중 어느 UNet 레이어가 SDD 에 가장 좋은 표현을 제공하는지 비교합니다.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train
import pandas as pd
from pathlib import Path

base_cfg = load_cfg("configs/cifar10.yaml")
base_cfg = deep_update(base_cfg, {
    "wandb": {"enabled": True, "tags": ["cifar10", "feature_layer_ablation"]},
})

layers = ["bottleneck", "skip1", "skip2", "decoder1"]
rows = []

for layer in layers:
    cfg = deep_update(base_cfg, {"sdd": {"feature_layer": layer}})
    cfg["run_name"] = f"cifar10_feat_{layer}"
    print(f"\n▶ feature_layer={layer}")
    launch_train(cfg, num_processes=None, with_eval=True)

    csv = Path(f"outputs/logs/cifar10_feat_{layer}_history.csv")
    if csv.exists():
        df = pd.read_csv(csv)
        last = df.dropna(subset=["val_fid"]).iloc[-1] if "val_fid" in df.columns else df.iloc[-1]
        rows.append({"feature_layer": layer,
                     "fid": last.get("val_fid"),
                     "linear_probe_acc": last.get("val_linear_probe_acc")})

results = pd.DataFrame(rows)
results


In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    r_fid = results.sort_values("fid")
    axes[0].barh(r_fid["feature_layer"], r_fid["fid"], color="steelblue")
    axes[0].set_xlabel("FID ↓"); axes[0].set_title("FID by feature layer")

    r_acc = results.sort_values("linear_probe_acc", ascending=False)
    axes[1].barh(r_acc["feature_layer"], r_acc["linear_probe_acc"], color="darkorange")
    axes[1].set_xlabel("Linear probe acc ↑"); axes[1].set_title("Probe acc by feature layer")

    plt.tight_layout()
    plt.savefig("outputs/figures/feature_layer_ablation.png", dpi=150)
    plt.show()

    print("Best FID layer:     ", results.loc[results["fid"].idxmin(), "feature_layer"])
    print("Best probe-acc layer:", results.loc[results["linear_probe_acc"].idxmax(), "feature_layer"])


**핵심 확인 포인트:** FID 최적 레이어와 probe-acc 최적 레이어가 다르면, 생성 품질과 표현력에 최적인 distillation 타겟이 다르다는 것을 보여주는 새로운 발견입니다.